In [4]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import RandomizedSearchCV

import warnings
warnings.filterwarnings('ignore')

In [5]:
df = pd.read_csv("Vehicle_Collision.csv")
df.head()

,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,PERSON_ID,PERSON_TYPE,PERSON_INJURY,VEHICLE_ID,PERSON_AGE,EJECTION,...,BODILY_INJURY,POSITION_IN_VEHICLE,SAFETY_EQUIPMENT,PED_LOCATION,PED_ACTION,COMPLAINT,PED_ROLE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2,PERSON_SEX
0,10253606,4230743.0,10/24/2019,19:15,84bcb3a7-d201-4c61-9e30-fe29268c1074,Occupant,Injured,19143343.0,27.0,Not Ejected,...,Back,Driver,Lap Belt & Harness,NaN,NaN,Complaint of Pain or Nausea,Driver,NaN,NaN,M
1,10248708,4229547.0,10/26/2019,1:15,a8904763-2870-42f3-865c-b53d8e5156e2,Pedestrian,Injured,NaN,24.0,NaN,...,Shoulder - Upper Arm,NaN,NaN,Pedestrian/Bicyclist/Other Pedestrian at Inter...,Crossing With Signal,None Visible,Pedestrian,Unspecified,Unspecified,F
2,10254556,4230715.0,10/26/2019,8:50,1a085543-ae9d-4a69-8cd0-fb7b33380a8b,Bicyclist,Injured,19143832.0,42.0,Not Ejected,...,Knee-Lower Leg Foot,Driver,Unknown,NaN,NaN,None Visible,Driver,NaN,NaN,M
3,10250834,4230376.0,10/26/2019,19:40,f58fc41f-497e-4f30-bf49-a9a80adec8b2,Bicyclist,Injured,19141949.0,36.0,Not Ejected,...,Back,Driver,NaN,NaN,NaN,Internal,Driver,NaN,NaN,M
4,10252474,4229773.0,10/26/2019,16:50,4bf13d12-8d7a-4cb0-997f-dd6a8b1adca6,Occupant,Injured,19142773.0,50.0,Not Ejected,...,Head,"Front passenger, if two or more persons, inclu...",Lap Belt & Harness,NaN,NaN,Complaint of Pain or Nausea,Passenger,NaN,NaN,F


In [6]:
df = df.dropna(subset=['PERSON_INJURY'])

In [7]:
drop_cols = [
    'UNIQUE_ID',
    'COLLISION_ID',
    'PERSON_ID',
    'VEHICLE_ID'
]

df = df.drop(columns=drop_cols)

In [8]:
X = df.drop('PERSON_INJURY', axis=1)

y = df['PERSON_INJURY'].map({
    'Injured': 0,
    'Killed': 1
})

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
num_cols = X_train.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

cat_cols = X_train.select_dtypes(
    include='object'
).columns.tolist()

In [11]:
for col in cat_cols:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

In [12]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [13]:
print(X_train.shape)
print(X_test.shape)

print("Numerical Columns:", len(num_cols))
print("Categorical Columns:", len(cat_cols))

(184524, 16)
(46132, 16)
Numerical Columns: 1
Categorical Columns: 15


In [14]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        class_weight='balanced',
        random_state=42
    ))
])

In [15]:
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [10, 20, 30, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

In [16]:
random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_grid,
    n_iter=3,
    cv=3,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

In [17]:
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median'))]),
                                                                               ['PERSON_AGE']),
                                                                              ('cat',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='most_frequent')),
                                                                                               ('encoder',
                                                                                                OneHotEncoder(handle_unknown='ignore'))]),
                                                                               ['CRASH_DATE',
                                                                                'CRASH_TIME',
                                                                                'PERSON_TYP...
                                                                                'CONTRIBUTING_FACTOR_2',
                                                                                'PERSON_SEX'])])),
                                             ('classifier',
                                              RandomForestClassifier(class_weight='balanced',
                                                                     random_state=42))]),
                   n_iter=3, n_jobs=-1,
                   param_distributions={'classifier__max_depth': [10, 20, 30,
                                                                  None],
                                        'classifier__min_samples_leaf': [1, 2,
                                                                         4],
                                        'classifier__min_samples_split': [2, 5,
                                                                          10],
                                        'classifier__n_estimators': [100, 200,
                                                                     300]},
                   random_state=42, scoring='f1')

In [18]:
print("Best Parameters:")
print(random_search.best_params_)

Best Parameters:
{'classifier__n_estimators': 300, 'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 4, 'classifier__max_depth': 30}


In [19]:
print("Best F1 Score:")
print(random_search.best_score_)

Best F1 Score:
0.062333604697578836


In [20]:
best_model = random_search.best_estimator_

In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

y_pred = best_model.predict(X_test)

In [22]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision :", precision_score(y_test, y_pred))
print("Recall :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

Accuracy : 0.8999176276771005
Precision : 0.029838022165387893
Recall : 0.6829268292682927
F1 Score : 0.05717786399836634


In [23]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.90      0.95     45927
           1       0.03      0.68      0.06       205

    accuracy                           0.90     46132
   macro avg       0.51      0.79      0.50     46132
weighted avg       0.99      0.90      0.94     46132



## Hyperparameter Tuning Summary

- RandomizedSearchCV was used to optimize Random Forest hyperparameters.
- Cross-validation was performed during tuning.
- The best parameter combination was selected based on F1-score.
- The tuned model outperformed the baseline model and was selected as the final candidate model.

In [24]:
best_model = random_search.best_estimator_

In [25]:
import joblib

joblib.dump(best_model, "accident_severity_pipeline.pkl")

['accident_severity_pipeline.pkl']

In [26]:
import joblib

loaded_model = joblib.load("accident_severity_pipeline.pkl")

print(type(loaded_model))

<class 'sklearn.pipeline.Pipeline'>


In [27]:
import pandas as pd

sample_data = pd.DataFrame({
    'CRASH_DATE': ['01/01/2024'],
    'CRASH_TIME': ['10:30'],
    'PERSON_TYPE': ['Driver'],
    'PERSON_AGE': [35],
    'EJECTION': ['Not Ejected'],
    'EMOTIONAL_STATUS': ['Conscious'],
    'BODILY_INJURY': ['Back Injury'],
    'POSITION_IN_VEHICLE': ['Driver Seat'],
    'SAFETY_EQUIPMENT': ['Seat Belt'],
    'PED_LOCATION': ['Unknown'],
    'PED_ACTION': ['Unknown'],
    'COMPLAINT': ['Pain'],
    'PED_ROLE': ['None'],
    'CONTRIBUTING_FACTOR_1': ['Driver Inattention'],
    'CONTRIBUTING_FACTOR_2': ['Unspecified'],
    'PERSON_SEX': ['M']
})

In [28]:
prediction = loaded_model.predict(sample_data)
print(prediction)

[0]


In [29]:
best_model.predict(sample_data)

array([0])

In [30]:
import joblib

model_path = r"C:\Users\Dell\Desktop\Accident_Severity_Prediction\accident_severity_pipeline.pkl"

joblib.dump(best_model, model_path)

print("Model saved successfully!")

Model saved successfully!


In [31]:
test_model = joblib.load(model_path)

print(type(test_model))

test_prediction = test_model.predict(sample_data)

print("Prediction after saving and loading:", test_prediction)

<class 'sklearn.pipeline.Pipeline'>
Prediction after saving and loading: [0]


In [32]:
import os

print(os.path.exists(model_path))

True


In [33]:
import sklearn
print(sklearn.__version__)

1.5.2
